# TUGAS PRAKTIKUM 5 – KONKURENSI DAN SINKRONISASI

**Nama:** Habil Arrafi  
**NIM:** 2411511014  

Notebook ini berisi penyelesaian empat latihan: penggunaan `RLock` pada rekening bank, simulasi deadlock tiga lock dan tiga thread, eksperimen bounded buffer dengan dua producer dan dua consumer, serta refleksi singkat.

> Jalankan sel secara berurutan. Output dapat sedikit berbeda pada setiap eksekusi karena penjadwalan thread oleh sistem operasi.

## 1. RLock pada Class RekeningBank

`RLock` (reentrant lock) mengizinkan thread yang sama mengambil lock yang sudah dimilikinya lebih dari satu kali. Hal ini diperlukan karena `transfer()` mengunci rekening asal, lalu memanggil `tarik()` yang juga mengunci rekening asal. Jika menggunakan `Lock` biasa, thread akan menunggu lock miliknya sendiri sehingga terjadi deadlock.

Pada contoh ini, rekening tujuan juga dikunci saat saldo ditambah. Untuk mencegah deadlock ketika dua rekening saling melakukan transfer secara bersamaan, urutan penguncian kedua rekening dibuat konsisten berdasarkan identitas objek.

In [1]:
import threading

class RekeningBank:
    def __init__(self, nama, saldo=0):
        self.nama = nama
        self.saldo = saldo
        self.lock = threading.RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setor harus lebih dari 0.")
        with self.lock:
            self.saldo += jumlah
            print(f"{self.nama}: setor Rp{jumlah:,} | saldo Rp{self.saldo:,}")

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah tarik harus lebih dari 0.")
        with self.lock:
            if self.saldo < jumlah:
                raise ValueError(f"Saldo {self.nama} tidak mencukupi.")
            self.saldo -= jumlah
            print(f"{self.nama}: tarik Rp{jumlah:,} | saldo Rp{self.saldo:,}")

    def transfer(self, rekening_tujuan, jumlah):
        if rekening_tujuan is self:
            raise ValueError("Tidak dapat transfer ke rekening sendiri.")
        # Urutan lock konsisten untuk mengurangi risiko deadlock antar rekening.
        rekening_pertama, rekening_kedua = sorted(
            [self, rekening_tujuan], key=id
        )
        with rekening_pertama.lock:
            with rekening_kedua.lock:
                self.tarik(jumlah)                 # Memanggil lock self lagi
                rekening_tujuan.setor(jumlah)     # Memanggil lock tujuan lagi
                print(f"Transfer Rp{jumlah:,}: {self.nama} -> {rekening_tujuan.nama}")

rekening_a = RekeningBank("Rekening A", 1_000_000)
rekening_b = RekeningBank("Rekening B", 500_000)

rekening_a.setor(200_000)
rekening_a.tarik(100_000)
rekening_a.transfer(rekening_b, 300_000)

print("\nSaldo akhir:")
print(rekening_a.nama, "=", f"Rp{rekening_a.saldo:,}")
print(rekening_b.nama, "=", f"Rp{rekening_b.saldo:,}")

Rekening A: setor Rp200,000 | saldo Rp1,200,000
Rekening A: tarik Rp100,000 | saldo Rp1,100,000
Rekening A: tarik Rp300,000 | saldo Rp800,000
Rekening B: setor Rp300,000 | saldo Rp800,000
Transfer Rp300,000: Rekening A -> Rekening B

Saldo akhir:
Rekening A = Rp800,000
Rekening B = Rp800,000


**Penjelasan:** `RLock` dibutuhkan karena `transfer()` memanggil `tarik()` pada objek rekening yang sama ketika lock rekening asal masih dipegang. `tarik()` kemudian mengambil lock tersebut kembali. `RLock` mengizinkan penguncian berulang oleh thread pemiliknya, sedangkan `Lock` biasa tidak. Penguncian kedua rekening dengan urutan yang konsisten juga membantu mencegah dua transfer berlawanan arah saling menunggu.

## 2. Deadlock dengan 3 Lock dan 3 Thread

Skenario dibuat melingkar: Thread 1 memegang Lock-A dan menunggu Lock-B; Thread 2 memegang Lock-B dan menunggu Lock-C; Thread 3 memegang Lock-C dan menunggu Lock-A. Ketiganya membentuk circular wait.

Agar notebook tidak berhenti selamanya, simulasi dijalankan sebagai proses terpisah dan dibatasi menggunakan `timeout`. Jika deadlock terjadi, proses dihentikan setelah batas waktu.

In [2]:
import subprocess
import sys
import textwrap

program_deadlock = r'''
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()
barrier = threading.Barrier(3)

def worker(nama, lock_awal, lock_berikutnya, nama_awal, nama_berikutnya):
    with lock_awal:
        print(f"{nama} memegang {nama_awal}", flush=True)
        barrier.wait()
        print(f"{nama} menunggu {nama_berikutnya}", flush=True)
        lock_berikutnya.acquire()
        print(f"{nama} berhasil mendapat {nama_berikutnya}", flush=True)
        lock_berikutnya.release()

threads = [
    threading.Thread(target=worker, args=("Thread 1", lock_a, lock_b, "Lock-A", "Lock-B")),
    threading.Thread(target=worker, args=("Thread 2", lock_b, lock_c, "Lock-B", "Lock-C")),
    threading.Thread(target=worker, args=("Thread 3", lock_c, lock_a, "Lock-C", "Lock-A")),
]

for t in threads:
    t.start()
for t in threads:
    t.join()
print("Semua thread selesai.", flush=True)
'''

try:
    hasil = subprocess.run(
        [sys.executable, "-u", "-c", program_deadlock],
        capture_output=True, text=True, timeout=3
    )
    print(hasil.stdout)
    print("Proses selesai tanpa timeout.")
except subprocess.TimeoutExpired as e:
    keluaran = e.stdout or ""
    if isinstance(keluaran, bytes):
        keluaran = keluaran.decode("utf-8", errors="replace")
    print(keluaran)
    print("TIMEOUT: proses dihentikan setelah 3 detik karena thread saling menunggu (deadlock).")


TIMEOUT: proses dihentikan setelah 3 detik karena thread saling menunggu (deadlock).


**Pembuktian dan penjelasan:** Setiap thread berhasil memegang satu lock, kemudian meminta lock berikutnya yang sedang dipegang thread lain. Tidak ada thread yang dapat melanjutkan dan melepaskan lock, sehingga proses tidak selesai sampai batas `timeout` tercapai. Penggunaan proses terpisah membuat simulasi aman dijalankan tanpa membuat kernel notebook ikut macet.

Pola ini memenuhi kondisi circular wait. Deadlock dapat dicegah, misalnya dengan menetapkan urutan global pengambilan lock (A lalu B lalu C), atau menggunakan mekanisme timeout saat mencoba memperoleh lock.

## 3. Bounded Buffer dengan Kapasitas Berbeda

Eksperimen menggunakan dua producer dan dua consumer. Masing-masing producer menghasilkan lima data. Consumer mengambil total sepuluh data, sehingga seluruh data yang diproduksi akan dikonsumsi. Percobaan diulang untuk kapasitas buffer 1, 3, dan 10.

`queue.Queue` menyediakan operasi thread-safe dan membatasi jumlah item sesuai kapasitas. Log menunjukkan urutan producer memasukkan data dan consumer mengambil data.

In [3]:
import threading
import queue
import time

def eksperimen_buffer(kapasitas):
    buffer = queue.Queue(maxsize=kapasitas)
    log = []
    log_lock = threading.Lock()
    jumlah_per_producer = 5
    jumlah_consumer = 2
    total_data = 2 * jumlah_per_producer
    sentinel = object()

    def catat(pesan):
        with log_lock:
            log.append(pesan)

    def producer(nama, awalan):
        for i in range(1, jumlah_per_producer + 1):
            item = f"{awalan}{i}"
            buffer.put(item)  # Menunggu jika buffer penuh
            catat(f"{nama} menghasilkan {item} (isi buffer: {buffer.qsize()})")

    def consumer(nama):
        while True:
            item = buffer.get()  # Menunggu jika buffer kosong
            if item is sentinel:
                buffer.task_done()
                break
            catat(f"{nama} mengambil {item} (isi buffer: {buffer.qsize()})")
            buffer.task_done()
            time.sleep(0.005)

    producers = [
        threading.Thread(target=producer, args=("Producer 1", "P1-")),
        threading.Thread(target=producer, args=("Producer 2", "P2-")),
    ]
    consumers = [
        threading.Thread(target=consumer, args=("Consumer 1",)),
        threading.Thread(target=consumer, args=("Consumer 2",)),
    ]

    for t in consumers + producers:
        t.start()
    for t in producers:
        t.join()

    # Dua sentinel digunakan agar kedua consumer dapat berhenti.
    for _ in range(jumlah_consumer):
        buffer.put(sentinel)
    for t in consumers:
        t.join()

    print(f"\nKAPASITAS BUFFER = {kapasitas}")
    for baris in log:
        print(baris)
    print(f"Total data diproduksi: {total_data}")
    print(f"Total data dikonsumsi: {sum(' mengambil ' in x for x in log)}")

for ukuran in (1, 3, 10):
    eksperimen_buffer(ukuran)


KAPASITAS BUFFER = 1
Producer 1 menghasilkan P1-1 (isi buffer: 1)
Consumer 1 mengambil P1-1 (isi buffer: 0)
Producer 1 menghasilkan P1-2 (isi buffer: 1)
Consumer 2 mengambil P1-2 (isi buffer: 0)
Producer 1 menghasilkan P1-3 (isi buffer: 1)
Consumer 1 mengambil P1-3 (isi buffer: 0)
Producer 1 menghasilkan P1-4 (isi buffer: 1)
Consumer 2 mengambil P1-4 (isi buffer: 0)
Producer 2 menghasilkan P2-1 (isi buffer: 1)
Consumer 1 mengambil P2-1 (isi buffer: 0)
Producer 1 menghasilkan P1-5 (isi buffer: 1)
Consumer 2 mengambil P1-5 (isi buffer: 0)
Producer 2 menghasilkan P2-2 (isi buffer: 1)
Consumer 1 mengambil P2-2 (isi buffer: 0)
Producer 2 menghasilkan P2-3 (isi buffer: 1)
Consumer 2 mengambil P2-3 (isi buffer: 0)
Producer 2 menghasilkan P2-4 (isi buffer: 1)
Consumer 1 mengambil P2-4 (isi buffer: 0)
Producer 2 menghasilkan P2-5 (isi buffer: 1)
Consumer 2 mengambil P2-5 (isi buffer: 0)
Total data diproduksi: 10
Total data dikonsumsi: 10

KAPASITAS BUFFER = 3
Producer 1 menghasilkan P1-1 (isi 

**Analisis hasil:** Pada kapasitas 1, producer dan consumer harus bergantian lebih sering karena hanya satu item yang dapat berada di buffer. Akibatnya, proses cenderung terlihat lebih berselang-seling. Pada kapasitas 3, producer dapat memasukkan beberapa item sebelum consumer mengambilnya, sehingga interleaving lebih bervariasi. Pada kapasitas 10, seluruh data dari kedua producer berpotensi masuk lebih dahulu sebelum semuanya diambil consumer, sehingga aktivitas producer dapat terlihat lebih berkelompok.

Ukuran buffer memengaruhi frekuensi producer harus menunggu ketika buffer penuh dan consumer harus menunggu ketika buffer kosong. Namun, urutan tepat setiap baris tidak selalu sama karena dipengaruhi penjadwalan thread. Kapasitas buffer yang lebih besar tidak otomatis membuat proses lebih cepat; pengaruhnya bergantung pada kecepatan producer dan consumer.

## 4. Refleksi Singkat (Maksimal 150 Kata)

Menurut saya, deadlock merupakan masalah yang paling berbahaya untuk aplikasi produksi nyata, terutama pada sistem perbankan dan e-commerce. Ketika deadlock terjadi, proses yang membutuhkan sumber daya tertentu dapat berhenti saling menunggu. Akibatnya, transaksi bisa tertunda dan layanan menjadi tidak responsif. Sementara itu, starvation juga berbahaya karena ada proses yang terus-menerus tidak memperoleh kesempatan menggunakan sumber daya. Livelock membuat proses tetap aktif, tetapi tidak menghasilkan kemajuan yang berarti.

Masalah yang paling sulit dideteksi saat testing menurut saya adalah starvation. Hal ini karena program masih berjalan dan tidak selalu menampilkan error. Proses tertentu hanya mengalami penundaan yang panjang, dan kondisinya bisa muncul ketika beban sistem tinggi atau pola penjadwalan tertentu terjadi. Karena itu, pengujian perlu dilakukan dengan berbagai kondisi beban dan pemantauan waktu tunggu setiap proses.

## Kesimpulan

1. `RLock` sesuai digunakan ketika method pada objek yang sama saling memanggil dan mengambil lock yang sama.
2. Deadlock tiga thread dapat dibuktikan melalui pola circular wait: setiap thread memegang satu lock dan menunggu lock berikutnya.
3. Kapasitas bounded buffer memengaruhi pola pergantian producer dan consumer, tetapi urutan interleaving tetap dipengaruhi penjadwalan thread.
4. Deadlock, livelock, dan starvation perlu dicegah melalui desain sinkronisasi dan pengujian pada berbagai kondisi beban.